# Supp Table 6 — architecture ablation (pooled)

**🔴 heavy (GPU / multi-GB / long)** · source: `notebooks/ablation_primary_models.py`

🔴 Trains 7 arms on GPU. The run cell is commented out; cached CSV shown.

## Configuration — edit the paths, then run

In [ ]:
import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
warnings.filterwarnings("ignore")
try:
    import scanpy as sc
    import anndata as ad
except Exception as e:
    print("[note] scanpy/anndata not available:", e)

# ── EDIT THESE PATHS to match your environment ──
REPO_ROOT     = Path("/path/to/spatnic")          # this repository
BACKUP_ROOT   = Path("/path/to/backup")           # integrate_adata_filtered.h5ad, galaxy scores, Liver meta
DATA_ROOT     = Path("/path/to/data")             # GxD concat, lung annotated, c2l refs, spatnic_models, GxD_Xenium
BENCHMARK_DB  = Path("/path/to/benchmark_db")     # Xenium/VisiumHD/MERFISH/CosMx + adata_hvg_*
VISIUMHD_ROOT = Path("/path/to/VisiumHD")         # Visium HD ADC track
WEIGHTS_DIR   = Path.home() / ".spatnic" / "weights"

# ── Derived ──
NB     = REPO_ROOT / "notebooks"
COMP   = NB / "comparison_results"
VHD    = VISIUMHD_ROOT
MODELS = DATA_ROOT / "spatnic_models"
PAPER  = REPO_ROOT / "paper"
BASE_DIR  = VHD          # VisiumHD ADC notebook global
THRESHOLD = 0.9          # overridden to 0.5 by the Fig 5 shortcut setup cell
sys.path[:0] = [str(REPO_ROOT / "scripts"), str(NB)]
if NB.exists():
    os.chdir(NB)         # extracted cells were written for cwd = notebooks/

def _tbl(csv, n=None):
    p = Path(csv)
    if not p.exists():
        print("[missing]", p); return None
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p)
    display(df.head(n) if n else df); return df

def _run(script, show=None, n=None):
    import subprocess
    cmd = f"python notebooks/{script}"
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=str(REPO_ROOT), capture_output=True, text=True)
    print((r.stdout or "")[-3000:])
    if r.returncode: print("STDERR:\n", (r.stderr or "")[-2000:])
    if show: _tbl(COMP / show, n)


## Regenerate (runs the real metric program)

In [ ]:
# _run("ablation_primary_models.py")   # 🔴 uncomment to retrain (GPU, long-running)
_tbl(COMP / "ablation_primary/metrics_pooled.csv")

## Result (current cached values)

**Ablation pooled** (`metrics_pooled.csv`, 12 rows)

| test_set | method | n_cells | auc_pr | auc_roc | f1 | mcc | accuracy | balanced_acc |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| internal | Student-t VAE (SPATNIC) | 97804 | 0.9905576516433884 | 0.9911598271055988 | 0.9611136112588472 | 0.9222528660568942 | 0.9611263343012556 | 0.9611263343012556 |
| internal | Gaussian VAE | 97804 | 0.9638317267285368 | 0.967313559703442 | 0.9112306797407346 | 0.825647583116849 | 0.9126211606887245 | 0.9126211606887245 |
| internal | MLP (supervised) | 97804 | 0.9737483108742956 | 0.9804096722774092 | 0.9474233235495648 | 0.8948714743806038 | 0.9474356877019344 | 0.9474356877019344 |
| internal | Random Forest | 97804 | 0.9667999050707464 | 0.9709524150216052 | 0.86606021187309 | 0.7248800454168333 | 0.8462946300764795 | 0.8462946300764795 |
| internal | Logistic Regression | 97804 | 0.9797440409286048 | 0.984466100423172 | 0.9433974084554102 | 0.8899134800934354 | 0.9445728191076028 | 0.944572819107603 |
| internal | XGBoost | 97804 | 0.975273132480494 | 0.9786967901159104 | 0.922232844257866 | 0.8473949434881761 | 0.923479612285796 | 0.923479612285796 |
| external | Student-t VAE (SPATNIC) | 673701 | 0.9967002194158748 | 0.9840557376709168 | 0.9694634058336676 | 0.8252621990375031 | 0.9492489991850984 | 0.9458351020250368 |
| external | Gaussian VAE | 673701 | 0.9814338329870708 | 0.9201909275367242 | 0.8623138508540861 | 0.5065104940275358 | 0.7907543554187986 | 0.8304923306391859 |
| external | MLP (supervised) | 673701 | 0.9741291269397476 | 0.9129988939617092 | 0.9347989545564636 | 0.6472134660309827 | 0.8929109501099153 | 0.8631711402690243 |
| external | Random Forest | 673701 | 0.9761505814954772 | 0.9122659008658436 | 0.9578003051526746 | 0.6914714895195754 | 0.9266365939786344 | 0.799409857916779 |
| external | Logistic Regression | 673701 | 0.9844323605800958 | 0.9453610016215296 | 0.9491509885060776 | 0.7172487288148469 | 0.9160250615629189 | 0.8957428920094235 |
| external | XGBoost | 673701 | 0.982229966863195 | 0.9250172567693812 | 0.8850165381425807 | 0.5365705653160785 | 0.8209472748296351 | 0.8385313718710969 |